In [ ]:
import pandas as pd
import re
import sys

In [ ]:
# Data sources
files = {
    "nat_uv_db":"nat_uv_db_processed.csv",
    "ANPDB":"ANPDB_processed.csv",
    "EMNPD":"EMNPD_processed.csv"
}

In [ ]:
# Sources metadata columns

source_configs = {
    "nat_uv_db": {
        "NP_name": "Common name",
        "PubChem_ID": "PubChem ID",
        "Species": "Species",
        "Reference": "DOI",
        "SMILES": "Canonical_SMILES",
        "InChIKey": "InChIKey",
        "canonical_smiles":"canonical_smiles",
        "non_activity_cols": [
           'Canonical_SMILES', 'NatUV ID', 'Common name', 'PubChem ID', 'Species',
           'DOI', 'canonical_smiles', 'InChIKey'
        ]
    },
    "ANPDB": {
        "NP_name": "mol_name",
        "PubChem_ID": "pubchem_id",
        "Species": "source_species",
        "Reference": "pmids",
        "SMILES": "smiles",
        "InChIKey": "InChIKey",
        "canonical_smiles":"canonical_smiles",
        "non_activity_cols": [
            'molecule_id', 'mol_name', 'pubchem_id', 'smiles', 'source_species',
            'pmids', 'links', 'bio_list', 'canonical_smiles', 'InChIKey'
        ]
    },
    "EMNPD": {
        "NP_name": "NPs_Name",
        "PubChem_ID": "PubChem_CID",
        "Species": "Endophyte_Name",
        "Reference": "PMID",
        "SMILES": "Smiles",
        "InChIKey": "InChIKey",
        "canonical_smiles":"canonical_smiles",
        "non_activity_cols": [
            'ENC_ID', 'NPs_Name', 'Smiles', 'PubChem_CID', 'Endophyte_Name',
            'PMID', 'canonical_smiles', 'InChIKey'
        ]
    }
}


In [ ]:
# Bioactivity dictioary

activity_mapping = {
    # Anticancer

    "anticancer": "Anticancer",
    "antitumor": "Anticancer",
    "antileukemia": "Anticancer",
    "antimigratory": "Anticancer",
    "anti-angiogenic activity": "Anticancer",
    "anti-proliferative activity": "Anticancer",
    "cell proliferation inhibition": "Anticancer",
    "antiproliferative": "Anticancer",


    #Cytotoxic
    "cytotoxic activity": "Cytotoxic",
    "cytotoxic": "Cytotoxic",


    "Anticancer / Citotoxic": ["Anticancer", "Cytotoxic"],

    # Antibacterial
    "antibacterial": "Antibacterial",
    "anti-bacterial activity": "Antibacterial",
    "anti-bacterial photodynamic activity": "Antibacterial",

    # Antifungal
    "antifungal": "Antifungal",
    "anti-fungal activity": "Antifungal",

    # Antimicrobial
    "antimicrobial": "Antimicrobial",
    "anti-microbial activity": "Antimicrobial",
    "anti-biofilm activity": "Antimicrobial",
    "anti-quorum sensing activity": "Antimicrobial",
    "anti-tuberculosis activity": "Antimicrobial",

    # Antiviral
    "antiviral": "Antiviral",
    "anti-viral activity": "Antiviral",
    "anti-respiratory syncytial virus (rsv)": "Antiviral",
    "anti-tobacco mosaic virus activity": "Antiviral",
    "neuraminidase inhibitory activity": "Antiviral",


    # Antiparasitic
    "antiparasitic": "Antiparasitic",
    "anti-parasitic activity": "Antiparasitic",
    "anti-leishmanial": "Antiparasitic",
    "anti-leishmanial activity": "Antiparasitic",
    "anti-plasmodial": "Antiparasitic",
    "anti-malarial activity": "Antiparasitic",
    "anthelmintic": "Antiparasitic",
    "schistosomicidal activity": "Antiparasitic",
    "trypanocidal activity": "Antiparasitic",
    "motility-impairing": "Antiparasitic",

    # Anti-inflammatory
    "anti-inflammatory": "Anti-inflammatory",
    "anti-inflammatory activity": "Anti-inflammatory",
    "anti-neuroinflammatory activity": "Anti-inflammatory",
    "cox-2 inhibitory activity": "Anti-inflammatory",


    # Antioxidant
    "antioxidant": "Antioxidant",
    "anti-oxidant activity": "Antioxidant",
    "anti-lipid peroxidation activity": "Antioxidant",
    "lipoprotein oxidation inhibitory activity": "Antioxidant",

    # Metabolic
    "11-β hydroxysteroid dehydrogenase-1 inhibitory activity": "Metabolic",
    "metabolism": "Metabolic",
    "anti-diabetic activity": "Metabolic",
    "hypoglycemic activity": "Metabolic",
    "cholesterol efflux enhancing activity": "Metabolic",
    "anti-hyperlipidemic activity": "Metabolic",
    "glucose uptake activity": "Metabolic",
    "pancreatic lipase inhibitory activity": "Metabolic",
    "peroxisome proliferator-activated receptor inhibitory activity": "Metabolic",
    "α-amylase inhibitory activity": "Metabolic",
    "α-glucosidase inhibitory activity": "Metabolic",
    "aldose reductase inhibitory activity": "Metabolic",

    # Neurological
    "neurological diseases/targets": "Neurological",
    "neuroprotective activity": "Neurological",
    "psychoactive receptor inhibitory activity": "Neurological",
    "antinociceptive": "Neurological",
    "acetylcholinesterase inhibitory activity": "Neurological",
    "butyrylcholinesterase inhibitory activity": "Neurological",
    "monoamine oxidase inhibitory activity": "Neurological",
    "anticholinesterase": "Neurological",
    "tremorgenic activity": "Neurological",

    # Immunomodulatory
    "immunomodulator": "Immunomodulatory",
    "immunosuppressive activity": "Immunomodulatory",
    "suppressive on phagocytosis response upon activation with serum": "Immunomodulatory",
    "calcineurin inhibitory activity": "Immunomodulatory",
    "indoleamine-2, 3-dioxygenase inhibitory activity": "Immunomodulatory",
    "toll like receptor agonist activity": "Immunomodulatory",
    "nf-κb inhibitory activity": "Immunomodulatory",
    "tumor necrosis factor-α inhibitory activity": "Immunomodulatory",

    # Cardiovascular
    "cardiovascular": "Cardiovascular",
    "cardioprotective activity": "Cardiovascular",
    "vasodepressor": "Cardiovascular",
    "angiotensin converting enzyme inhibitory activity": "Cardiovascular",
    "anti-atherosclerosis activity": "Cardiovascular",
    "antiplatelet/anticoagulant": "Cardiovascular",
    "anti-platelet activity": "Cardiovascular",
    "anti-coagulant activity": "Cardiovascular",

    # Agrochemical
    "insecticide": "Agrochemical",
    "insecticidal activity": "Agrochemical",
    "nematicidal": "Agrochemical",
    "nematicidal activity": "Agrochemical",
    "phytotoxicity": "Agrochemical",
    "phytotoxic activity": "Agrochemical",
    "allelopathic activity": "Agrochemical",
    "anti-feedant activity": "Agrochemical",
    "anti-algal activity": "Agrochemical",
    "photosynthetic inhibition activity": "Agrochemical",
    "plant growth-promoting activity": "Agrochemical",
    "anti-fouling activity": "Agrochemical",

    # Antiallergic
    "antiallergic": "Antiallergic",
    "anti-allergic activity": "Antiallergic",

    # Toxic
    "toxin": "Toxic",
    "zootoxic": "Toxic",
    "hemolytic activity": "Toxic",

    # Anti-aging
    "anti-aging activity": "Anti-aging",


    # Enzyme/Target Inhibition
    "protein kinase inhibitory activity": "Enzyme/Target Inhibition",
    "protein kinase inhibition": "Enzyme/Target Inhibition",
    "kinase inhibitory": "Enzyme/Target Inhibition",
    "tyrosine kinase inhibitory activity": "Enzyme/Target Inhibition",
    "inhibitory toward p56lck tyrosine": "Enzyme/Target Inhibition",
    "phosphatidylinositol 3-kinase inhibitory activity": "Enzyme/Target Inhibition",
    "cyclin dependent kinase inhibitory activity": "Enzyme/Target Inhibition",
    "inhibition of the hsp90 machine chaperoning": "Enzyme/Target Inhibition",
    "hdac inhibition": "Enzyme/Target Inhibition",
    "isocitrate dehydrogenase 1 inhibitory activity": "Enzyme/Target Inhibition",
    "pyruvate dehydrogenase kinase inhibitory activity": "Enzyme/Target Inhibition",
    "topoisomerase inhibitory activity": "Enzyme/Target Inhibition",
    "xanthine oxidase inhibitory activity": "Enzyme/Target Inhibition",
    "phosphodiesterase inhibitory activity": "Enzyme/Target Inhibition",
    "urease inhibitory activity": "Enzyme/Target Inhibition",
    "anti-trypsin activity": "Enzyme/Target Inhibition",
    "protease inhibitory activity": "Enzyme/Target Inhibition",
    "proteasome inhibitory activity": "Enzyme/Target Inhibition",
    "carboxylesterases inhibitory activity": "Enzyme/Target Inhibition",
    "protein tyrosine phosphatase 1b inhibitory activity": "Enzyme/Target Inhibition",
    "β-glucuronidase inhibitory activity": "Enzyme/Target Inhibition",
    "adenosine deaminase inhibitory activity": "Enzyme/Target Inhibition",
    "biochemical induction activity": "Enzyme/Target Inhibition",
    "cystic fibrosis transmembrane conductance regulator inhibitory activity": "Enzyme/Target Inhibition",
    "anti-melanogenic activity": "Enzyme/Target Inhibition"
}


In [ ]:
def clean_label(x):
    x = str(x).strip().lower()
    x = x.replace("_", " ")
    x = re.sub(r"\s+", " ", x)
    return x

In [ ]:
all_compounds = []
all_activities_long = []

# Process each source

for source, path in files.items():

    df = pd.read_csv(path)
    cfg = source_configs[source]

    # Clean column names
    df.columns = [c.strip() for c in df.columns]

    # Detect activity columns
    non_activity_cols = [c for c in cfg["non_activity_cols"] if c in df.columns]
    activity_cols = [c for c in df.columns if c not in non_activity_cols]

    # Create standardized compound metadata table
    temp_compounds = pd.DataFrame()
    temp_compounds["source_database"] = source
    temp_compounds["NP_name"] = df[cfg["NP_name"]]
    temp_compounds["PubChem_ID"] = df[cfg["PubChem_ID"]]
    temp_compounds["Source_species"] = df[cfg["Species"]]
    temp_compounds["Reference"] = df[cfg["Reference"]]
    temp_compounds["SMILES"] = df[cfg["SMILES"]]
    temp_compounds["canonical_smiles"] = df[cfg["canonical_smiles"]]
    temp_compounds["InChIKey"] = df[cfg["InChIKey"]]

    all_compounds.append(temp_compounds)

    long_df = df.melt(
        id_vars=[cfg["InChIKey"]],
        value_vars=activity_cols,
        var_name="original_activity",
        value_name="activity_value"
    )

    # Create original activity and mapped activity matrix
    # Keep only exact integer 1
    long_df = long_df[long_df["activity_value"] == 1].copy()

    long_df["source_database"] = source
    long_df["InChIKey"] = long_df[cfg["InChIKey"]]
    long_df["original_activity_clean"] = long_df["original_activity"].apply(clean_label)
    long_df["final_activity"] = long_df["original_activity_clean"].map(activity_mapping)

    long_df = long_df.dropna(subset=["final_activity"])

    long_df["final_activity"] = long_df["final_activity"].apply(lambda x: x if isinstance(x, list) else [x])
    long_df = long_df.explode("final_activity")

    all_activities_long.append(long_df[["InChIKey", "source_database", "original_activity", "final_activity"]])


# Merge compounds and remove duplicates

compounds_all = pd.concat(all_compounds, ignore_index=True)

compounds_all = compounds_all.dropna(subset=["InChIKey"])
compounds_all = compounds_all[compounds_all["InChIKey"].astype(str).str.strip() != ""]

compounds_unique = (
    compounds_all
    .sort_values(["InChIKey", "source_database"])
    .groupby("InChIKey", as_index=False)
    .agg({
        "NP_name": lambda x: next((i for i in x if pd.notna(i) and i != ""), None),
        "PubChem_ID": lambda x: next((i for i in x if pd.notna(i) and i != ""), None),
        "Source_species": lambda x: "|| ".join(sorted(set(i for i in x if pd.notna(i) and i != ""))),
        "SMILES": lambda x: next((i for i in x if pd.notna(i) and i != ""), None),
        "canonical_smiles": lambda x: next((i for i in x if pd.notna(i) and i != ""), None),
        "Reference": lambda x: "|| ".join(sorted(set(str(i).strip() for i in x if pd.notna(i) and str(i).strip() != ""))),
       "source_database": lambda x: "||".join(sorted(set(str(i).strip() for i in x if pd.notna(i) and str(i).strip() != "")))
    })
)

compounds_unique.insert(
    0,
    "compound_id",
    [f"FNP{str(i+1).zfill(5)}" for i in range(len(compounds_unique))]
)

id_map = compounds_unique[["compound_id", "InChIKey"]]

# Build final multilabel bioactivity matrix

activities_long = pd.concat(all_activities_long, ignore_index=True)
activities_long = activities_long.dropna(subset=["InChIKey"])
activities_long = activities_long.merge(id_map, on="InChIKey", how="inner")

activities_long = activities_long.drop_duplicates(
    subset=["compound_id", "source_database", "original_activity", "final_activity"]
)

compound_label_pairs = activities_long[["compound_id", "final_activity"]].drop_duplicates()

bioactivities_multilabel = (
    compound_label_pairs
    .assign(value=1)
    .pivot_table(
        index="compound_id",
        columns="final_activity",
        values="value",
        fill_value=0,
        aggfunc="max"
    )
    .reset_index()
)

# Antimicrobial is the parent of Antibacterial and Antifungal
bioactivities_multilabel["Antimicrobial"] = (bioactivities_multilabel[
                                              ["Antimicrobial", "Antibacterial", "Antifungal"]].max(axis=1))

# Provenance table

bioactivity_provenance = (
    activities_long
    .groupby(["compound_id", "final_activity"])
    .agg({
        "source_database": lambda x: "||".join(sorted(set(x))),
        "original_activity": lambda x: "|| ".join(sorted(set(x)))
    })
    .reset_index()
    .rename(columns={
        "source_database": "supporting_sources",
        "original_activity": "original_activity_terms"
    })
)



In [ ]:
# Curation statistics

stats = []

for source in files:
    source_raw = pd.read_csv(files[source])
    stats.append({
        "step": f"Raw records in {source}",
        "count": len(source_raw)
    })

stats.extend([
    {"step": "Total raw records", "count": sum(len(pd.read_csv(p)) for p in files.values())},
    {"step": "Records with valid InChIKey", "count": len(compounds_all)},
    {"step": "Unique compounds after deduplication", "count": len(compounds_unique)},
    {"step": "Unique compound-bioactivity pairs", "count": len(compound_label_pairs)},
    {"step": "Final bioactivity labels", "count": len(bioactivities_multilabel.columns) - 1}
])

curation_statistics = pd.DataFrame(stats)

activity_mapping_df = pd.DataFrame([
    {"original_activity_clean": k, "final_activity": v}
    for k, v in activity_mapping.items()
])

In [ ]:
# Save outputs
compounds_unique.to_csv("compounds_unique.csv", index=False)
bioactivities_multilabel.to_csv("bioactivities_multilabel.csv", index=False)
bioactivity_provenance.to_csv("bioactivity_provenance.csv", index=False)
activities_long.to_csv("bioactivities_long_evidence.csv", index=False)
activity_mapping_df.to_csv("activity_mapping.csv", index=False)
curation_statistics.to_csv("curation_statistics.csv", index=False)

print("\nDone.")
print("Unique compounds:", len(compounds_unique))
print("Final labels:", list(bioactivities_multilabel.columns[1:]))


Done.
Unique compounds: 2304
Final labels: ['Agrochemical', 'Anti-aging', 'Anti-inflammatory', 'Antiallergic', 'Antibacterial', 'Anticancer', 'Antifungal', 'Antimicrobial', 'Antioxidant', 'Antiparasitic', 'Antiviral', 'Cardiovascular', 'Cytotoxic', 'Enzyme/Target Inhibition', 'Immunomodulatory', 'Metabolic', 'Neurological', 'Toxic']


In [42]:
bioactivity_df = pd.read_csv("bioactivities_multilabel.csv")
label_columns = bioactivity_df.columns[1:]

# Total bioactivity annotations (number of 1s)
total_annotations = int(bioactivity_df[label_columns].sum().sum())

# Average number of labels per fungal SM
average_labels = bioactivity_df[label_columns].sum(axis=1).mean()
print(f"Number of labels: {len(label_columns)}")
print(f"Average labels per SM: {average_labels:.2f}")
print(f"Total bioactivity annotations: {total_annotations:,}")

Number of labels: 18
Average labels per SM: 1.95
Total bioactivity annotations: 4,493


In [ ]:
import zipfile, os

In [ ]:
zip_path = "coconut_csv-05-2026.zip"
extract_path = "COCONUT_data"

if not os.path.exists(extract_path):
    zipfile.ZipFile(zip_path).extractall(extract_path)

In [ ]:
# Read (coconut DB)
coconut = pd.read_csv("/content/COCONUT_data/coconut_csv-05-2026.csv")
print(coconut.columns)

/tmp/ipykernel_1961/3190219558.py:2: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  coconut = pd.read_csv("/content/COCONUT_data/coconut_csv-05-2026.csv")


Index(['identifier', 'canonical_smiles', 'standard_inchi',
       'standard_inchi_key', 'name', 'iupac_name', 'annotation_level',
       'total_atom_count', 'heavy_atom_count', 'molecular_weight',
       'exact_molecular_weight', 'molecular_formula', 'alogp',
       'topological_polar_surface_area', 'rotatable_bond_count',
       'hydrogen_bond_acceptors', 'hydrogen_bond_donors',
       'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski',
       'lipinski_rule_of_five_violations', 'aromatic_rings_count',
       'qed_drug_likeliness', 'formal_charge', 'fractioncsp3',
       'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar',
       'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework',
       'np_likeness', 'chemical_class', 'chemical_sub_class',
       'chemical_super_class', 'direct_parent_classification',
       'np_classifier_pathway', 'np_classifier_superclass',
       'np_classifier_class', 'np_classifier_is_glycoside', 'organisms',


In [ ]:
# Keep only needed cols from COCONUT
coco_keep = ["np_classifier_pathway","standard_inchi_key"]
coconut_small = coconut[coco_keep]

In [ ]:
pathway_counts = (
    coconut_small.groupby("standard_inchi_key")["np_classifier_pathway"]
    .nunique()
)
print(pathway_counts.describe())

count    728421.000000
mean          0.930148
std           0.258104
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: np_classifier_pathway, dtype: float64


In [ ]:
multi_pathway = pathway_counts[pathway_counts > 1]

print(f"{len(multi_pathway)} InChIKeys have multiple pathways")

599 InChIKeys have multiple pathways


In [ ]:
# Check if fungal compounds have multiple pathway
my_ambiguous = compounds_unique[
    compounds_unique["InChIKey"].isin(multi_pathway)]

print("Ambiguous keys found in my dataset:", my_ambiguous["InChIKey"].nunique())

Ambiguous keys found in my dataset: 0


In [ ]:
# Reduce COCONUT to unique InChIKey
coconut_unique = (
    coconut_small
    .drop_duplicates(subset=["standard_inchi_key"])
    .copy()
)

In [ ]:
# Merge Np pathway onto metadata dataset
compound_metadata = compounds_unique.merge(coconut_unique, left_on="InChIKey",
    right_on="standard_inchi_key", how="left")

In [ ]:
# Count how many compounds have NP_Pathway filled
matched = compound_metadata["np_classifier_pathway"].notna().sum()
nulls   = compound_metadata["np_classifier_pathway"].isna().sum()
total   = len(compound_metadata)

print(f"Total compounds: {total}")
print(f"Matched with NP_Pathway: {matched}")
print(f"No NP_Pathway (null):   {nulls}")

Total compounds: 2304
Matched with NP_Pathway: 1490
No NP_Pathway (null):   814


In [ ]:
# Mark compounds with no NP_Pathway as Unknown
compound_metadata["np_classifier_pathway"] = compound_metadata["np_classifier_pathway"].fillna("Unknown")

In [ ]:
# Rename the "np_classifier_pathway" column to "NP_pathway"
compound_metadata = compound_metadata.rename(columns={"np_classifier_pathway": "NP_pathway"})

In [ ]:
# Drop "standard_inchi_key" column
compound_metadata = compound_metadata.drop(columns=["standard_inchi_key"])

In [ ]:
compound_metadata.to_csv("compound_metadata.csv", index=False)